## Bước 2: Clean CSV và load vào MySQL.
Chạy: python clean_load.ipynb
Yêu cầu: pip install pandas mysql-connector-python python-dotenv

In [1]:
import os
import pandas as pd
import mysql.connector
from mysql.connector import Error
from dotenv import load_dotenv

load_dotenv("./.env")

CSV_PATH = "../data/raw/ecommerce_sales_analytics_5000.csv"
OUTPUT_PATH = "../data/processed/orders_cleaned.csv"

DB_CONFIG = {
    "host": os.getenv("DB_HOST"),
    "user": os.getenv("DB_USER"),
    "password": os.getenv("DB_PASSWORD"),
    "database": os.getenv("DB_NAME"),
}

VALID_CATEGORIES = {"Beauty", "Clothing", "Electronics"}
VALID_REGIONS = {"North", "South", "East", "West"}
VALID_PAYMENTS = {"Wallet", "Card", "COD"}


def load_raw(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    print(f"[LOAD] Đọc {len(df)} dòng từ {path}")
    return df


def clean(df: pd.DataFrame) -> pd.DataFrame:
    original_len = len(df)

    # 1. Chuẩn hóa kiểu dữ liệu ngày tháng
    df["order_date"] = pd.to_datetime(df["order_date"], format="%m/%d/%Y", errors="coerce")

    # 2. Loại bỏ dòng thiếu các trường bắt buộc
    required_cols = ["order_id", "order_date", "customer_id", "product_category",
                      "region", "quantity", "unit_price", "revenue"]
    before = len(df)
    df = df.dropna(subset=required_cols)
    print(f"[CLEAN] Loại {before - len(df)} dòng thiếu trường bắt buộc")

    # 3. Xử lý missing ở trường không bắt buộc
    df["customer_rating"] = df["customer_rating"].fillna(df["customer_rating"].median())
    df["delivery_days"] = df["delivery_days"].fillna(df["delivery_days"].median()).astype(int)
    df["payment_method"] = df["payment_method"].fillna("Unknown")

    # 4. Loại bỏ trùng lặp theo order_id (giữ bản ghi đầu tiên)
    before = len(df)
    df = df.drop_duplicates(subset="order_id", keep="first")
    print(f"[CLEAN] Loại {before - len(df)} dòng trùng order_id")

    # 5. Loại bỏ giá trị âm / vô lý
    before = len(df)
    df = df[(df["quantity"] > 0) & (df["unit_price"] >= 0) & (df["revenue"] >= 0)]
    print(f"[CLEAN] Loại {before - len(df)} dòng có quantity/unit_price/revenue âm hoặc 0")

    # 6. Chuẩn hóa discount về khoảng [0, 1]
    df["discount"] = df["discount"].clip(lower=0, upper=1)

    # 7. Chuẩn hóa text categorical (viết hoa chữ cái đầu, loại khoảng trắng thừa)
    for col in ["product_category", "region", "payment_method"]:
        df[col] = df[col].astype(str).str.strip().str.title()

    # 8. Lọc giá trị categorical không hợp lệ (nếu có giá trị lạ sau chuẩn hóa)
    before = len(df)
    df = df[
        df["product_category"].isin(VALID_CATEGORIES) &
        df["region"].isin(VALID_REGIONS) &
        df["payment_method"].isin(VALID_PAYMENTS)
    ]
    print(f"[CLEAN] Loại {before - len(df)} dòng có category/region/payment không hợp lệ")

    # 9. Kiểm tra và sửa revenue nếu lệch công thức quá nhiều (giữ nguyên revenue gốc, chỉ cảnh báo)
    expected = df["quantity"] * df["unit_price"] * (1 - df["discount"])
    mismatch = (df["revenue"] - expected).abs() > 1.0
    print(f"[CLEAN] Cảnh báo: {mismatch.sum()} dòng có revenue lệch > 1.0 so với công thức tính")

    print(f"[CLEAN] Còn lại {len(df)}/{original_len} dòng sau khi clean "
          f"({len(df)/original_len*100:.1f}%)")

    return df.reset_index(drop=True)


def save_processed(df: pd.DataFrame, path: str):
    df.to_csv(path, index=False)
    print(f"[SAVE] Đã lưu dữ liệu sạch vào {path}")


def load_to_mysql(df: pd.DataFrame):
    conn = None
    try:
        conn = mysql.connector.connect(**DB_CONFIG)
        cursor = conn.cursor()

        insert_query = """
            INSERT INTO orders
            (order_id, order_date, customer_id, product_category, region,
             quantity, unit_price, discount, payment_method, delivery_days,
             customer_rating, revenue)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
            ON DUPLICATE KEY UPDATE
                revenue = VALUES(revenue),
                loaded_at = CURRENT_TIMESTAMP
        """

        rows = [
            (
                int(r.order_id),
                r.order_date.strftime("%Y-%m-%d"),
                int(r.customer_id),
                r.product_category,
                r.region,
                int(r.quantity),
                float(r.unit_price),
                float(r.discount),
                r.payment_method,
                int(r.delivery_days),
                float(r.customer_rating),
                float(r.revenue),
            )
            for r in df.itertuples(index=False)
        ]

        cursor.executemany(insert_query, rows)
        conn.commit()
        print(f"[MYSQL] Đã insert/update {cursor.rowcount} dòng vào bảng orders")

    except Error as e:
        print(f"[MYSQL] Lỗi: {e}")
        if conn:
            conn.rollback()
    finally:
        if conn and conn.is_connected():
            cursor.close()
            conn.close()


def main():
    df_raw = load_raw(CSV_PATH)
    df_clean = clean(df_raw)
    save_processed(df_clean, OUTPUT_PATH)
    load_to_mysql(df_clean)


if __name__ == "__main__":
    main()

[LOAD] Đọc 5000 dòng từ ../data/raw/ecommerce_sales_analytics_5000.csv
[CLEAN] Loại 0 dòng thiếu trường bắt buộc
[CLEAN] Loại 0 dòng trùng order_id
[CLEAN] Loại 0 dòng có quantity/unit_price/revenue âm hoặc 0
[CLEAN] Loại 2382 dòng có category/region/payment không hợp lệ
[CLEAN] Cảnh báo: 0 dòng có revenue lệch > 1.0 so với công thức tính
[CLEAN] Còn lại 2618/5000 dòng sau khi clean (52.4%)
[SAVE] Đã lưu dữ liệu sạch vào ../data/processed/orders_cleaned.csv
[MYSQL] Đã insert/update 2618 dòng vào bảng orders
